# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.shape
df.columns.tolist()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 33.1 MB


## A.2. Missing values & Duplicate data

In [3]:
print(df.isnull().sum())
print(df.duplicated().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
5268


## A.3. Invalid values

In [4]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [7]:
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df_clean['Sales'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [8]:
df_clean[['Quantity', 'UnitPrice', 'Sales']].mean()
df_clean[['Quantity', 'UnitPrice', 'Sales']].median()
df_clean[['Quantity', 'UnitPrice', 'Sales']].mode().iloc[0]


Quantity      1.00
UnitPrice     1.25
Sales        15.00
Name: 0, dtype: float64

## Group 2 — Dispersion

In [9]:
df_clean[['Quantity', 'UnitPrice', 'Sales']].std()
df_clean[['Quantity', 'UnitPrice', 'Sales']].var()

Quantity     24187.752994
UnitPrice     1289.936149
Sales        73092.768604
dtype: float64

## Group 3 — Location and Shape

In [10]:
df_clean[['Quantity', 'UnitPrice', 'Sales']].quantile([0.25, 0.5, 0.75])
df_clean[['Quantity', 'UnitPrice', 'Sales']].skew()
df_clean[['Quantity', 'UnitPrice', 'Sales']].kurtosis()

Quantity     236462.342826
UnitPrice     62483.142715
Sales        297651.661046
dtype: float64

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [11]:
country_sales = df_clean.groupby('Country')['Sales'].sum()
top_country = country_sales.idxmax()
pct = (country_sales.max() / country_sales.sum()) * 100
print(f"Quốc gia: {top_country}, Chiếm: {pct:.2f}%")

Quốc gia: United Kingdom, Chiếm: 84.61%


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [12]:
df_clean.groupby(['StockCode', 'Description'])['Sales'].sum().sort_values(ascending=True)

StockCode  Description                       
PADS       PADS TO MATCH ALL CUSHIONS                 0.003
84227      HEN HOUSE W CHICK IN NEST                  0.420
23366      SET 12 COLOURING PENCILS DOILEY            0.650
51014c     FEATHER PEN,COAL BLACK                     0.830
21268      VINTAGE BLUE TINSEL REEL                   0.840
                                                    ...    
47566      PARTY BUNTING                          99504.330
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.290
23843      PAPER CRAFT , LITTLE BIRDIE           168469.600
22423      REGENCY CAKESTAND 3 TIER              174484.740
DOT        DOTCOM POSTAGE                        206248.770
Name: Sales, Length: 4161, dtype: float64

## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [13]:
df_clean.set_index('InvoiceDate').resample('ME')['Sales'].sum()

InvoiceDate
2010-12-31     823746.140
2011-01-31     691364.560
2011-02-28     523631.890
2011-03-31     717639.360
2011-04-30     537808.621
2011-05-31     770536.020
2011-06-30     761739.900
2011-07-31     719221.191
2011-08-31     759138.380
2011-09-30    1058590.172
2011-10-31    1154979.300
2011-11-30    1509496.330
2011-12-31     638792.680
Freq: ME, Name: Sales, dtype: float64

## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [14]:

df_clean.groupby('Country').apply(
    lambda x : x['Sales'].sum() / x['InvoiceNo'].nunique(),
    include_groups=False
).sort_values(ascending=True)

Country
Saudi Arabia             145.920000
Bahrain                  251.380000
European Community       325.062500
Unspecified              365.368462
Poland                   386.034211
Czech Republic           413.370000
Lithuania                415.265000
Belgium                  420.370816
Italy                    460.085263
Germany                  500.803370
United Kingdom           500.872528
France                   534.987526
Malta                    545.118000
Finland                  549.904390
Portugal                 581.846552
Austria                  599.922353
Canada                   611.063333
Iceland                  615.714286
United Arab Emirates     634.093333
Spain                    684.190111
USA                      716.078000
Channel Islands          786.555385
Cyprus                   849.398750
Greece                   952.104000
EIRE                     984.215139
RSA                     1002.310000
Norway                  1004.595556
Israel              

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [15]:
(df['Quantity'] < 0).groupby(df['Country']).mean().mul(100).sort_values(ascending=False)

Country
USA                     38.487973
Czech Republic          16.666667
Malta                   11.811024
Japan                   10.335196
Saudi Arabia            10.000000
Australia                5.877681
Italy                    5.603985
Bahrain                  5.263158
Germany                  4.770932
EIRE                     3.684724
Poland                   3.225806
Singapore                3.056769
Sweden                   2.380952
Denmark                  2.313625
Spain                    1.894986
United Kingdom           1.855178
Belgium                  1.836636
Switzerland              1.748252
France                   1.741264
European Community       1.639344
Finland                  1.438849
Hong Kong                1.388889
Channel Islands          1.319261
Norway                   1.289134
Cyprus                   1.286174
Portugal                 1.184990
Austria                  0.748130
Greece                   0.684932
Israel                   0.673401
Nether

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*